In [ ]:
# Dzialwa Nemakonde
# 22 April 2026

# We want to create company brochurs by analysing the contents of a company's website
# To go into websites we use the fetch_website_contents from the web_scraper library

In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from web_scraper import fetch_website_contents

In [6]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key = openai_api_key)

In [7]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [8]:
def stream_gpt(prompt):
    
    messages = [{"role":"system", "content":system_message}, {"role":"user", "content":prompt}]
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages, 
        stream=True
        )
    
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [9]:
def stream_brochure(company_name, url):
    
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    yield from stream_gpt(prompt)


In [10]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
        ["Hugging Face", "https://huggingface.co/"],
        ["DM Academy", "https://www.dm-academy.app/"]
    ], 
    flagging_mode="never"
)


In [11]:
view.launch(inbrowser = True, share = True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e93b70755bfb2a28e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
